In [ ]:
# Imports
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configurar visualización
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## Paso 1: Construir Features

Antes de entrenar modelos, necesitamos features adecuadas:

```bash
python -m md_lakehouse.cli features --team configs/teams/team01.yaml --run-date 2026-02-01
```


In [ ]:
# Verificar que features fueron generadas
from pyspark.sql import SparkSession
from md_lakehouse.common.spark import get_spark_session
from md_lakehouse.common.io import read_parquet_snapshot

spark = get_spark_session(app_name="ML_Exploration")

ml_features = read_parquet_snapshot(spark, Path("data/gold"), "ml_features", "2026-02-01")

print(f"ML Features: {ml_features.count():,} filas")
print(f"\nColumnas ({len(ml_features.columns)}):")
for col in sorted(ml_features.columns):
    print(f"  - {col}")

In [ ]:
# Convertir a pandas para análisis exploratorio
pdf = ml_features.toPandas()

print("Estadísticas descriptivas:")
pdf.describe()

## Paso 2: Entrenar Modelos

Entrenamos los tres modelos:

```bash
python -m md_lakehouse.cli train --team configs/teams/team01.yaml --run-date 2026-02-01
```

Esto entrenará:
1. **Churn**: Random Forest Classifier
2. **LTV**: Random Forest Regressor
3. **Segmentation**: K-Means Clustering


## Paso 3: Analizar Modelo de Churn


In [ ]:
# Cargar métricas de churn
with open("data/models/churn_metrics_2026-02-01.json") as f:
    churn_metrics = json.load(f)

print("="*60)
print("MODELO DE CHURN - MÉTRICAS")
print("="*60)
print(f"Accuracy:  {churn_metrics['accuracy']:.4f}")
print(f"Precision: {churn_metrics['precision']:.4f}")
print(f"Recall:    {churn_metrics['recall']:.4f}")
print(f"F1 Score:  {churn_metrics['f1']:.4f}")
print(f"ROC AUC:   {churn_metrics['roc_auc']:.4f}")
print(f"\nSamples:")
print(f"  Train: {churn_metrics['train_samples']:,}")
print(f"  Test:  {churn_metrics['test_samples']:,}")

In [ ]:
# Visualizar matriz de confusión
import numpy as np

cm = np.array(churn_metrics['confusion_matrix'])

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Churn Model - Confusion Matrix')
plt.tight_layout()
plt.show()

# Calcular métricas de la matriz
tn, fp, fn, tp = cm.ravel()
print(f"\nTrue Negatives:  {tn:,}")
print(f"False Positives: {fp:,}")
print(f"False Negatives: {fn:,}")
print(f"True Positives:  {tp:,}")

In [ ]:
# Feature importance
top_features = pd.DataFrame(churn_metrics['top_features'][:10])

plt.figure(figsize=(10, 6))
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Importance')
plt.title('Top 10 Features - Churn Model')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### Interpretación del Modelo de Churn

**Pregunta**: ¿Qué features son más importantes para predecir churn?

**Análisis**:
- Las features de uso reciente (`last_7d_minutes`, `last_30d_minutes`) son típicamente muy importantes
- El `payment_fail_rate` indica problemas de pago que predicen churn
- `engagement_score` resume el nivel de actividad del usuario

**Acción de Negocio**:
- Usuarios con bajo uso reciente → campaña de reactivación
- Usuarios con payment_fail_rate alto → soporte de pagos
- Usuarios con bajo engagement → promos personalizadas


## Paso 4: Analizar Modelo de LTV


In [ ]:
# Cargar métricas de LTV
with open("data/models/ltv_metrics_2026-02-01.json") as f:
    ltv_metrics = json.load(f)

print("="*60)
print("MODELO DE LTV - MÉTRICAS")
print("="*60)
print(f"MAE (Mean Absolute Error): ${ltv_metrics['mae']:.2f}")
print(f"RMSE (Root Mean Squared Error): ${ltv_metrics['rmse']:.2f}")
print(f"R² Score: {ltv_metrics['r2']:.4f}")
print(f"\nPromedio:")
print(f"  LTV Real:      ${ltv_metrics['mean_actual']:.2f}")
print(f"  LTV Predicho:  ${ltv_metrics['mean_predicted']:.2f}")

In [ ]:
# Feature importance LTV
top_features_ltv = pd.DataFrame(ltv_metrics['top_features'][:10])

plt.figure(figsize=(10, 6))
plt.barh(range(len(top_features_ltv)), top_features_ltv['importance'])
plt.yticks(range(len(top_features_ltv)), top_features_ltv['feature'])
plt.xlabel('Importance')
plt.title('Top 10 Features - LTV Model')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### Interpretación del Modelo de LTV

**Pregunta**: ¿Qué predice el valor de vida del cliente?

**Análisis**:
- `tenure_days`: Clientes más antiguos tienen mayor LTV
- `avg_payment`: Pagos consistentes indican mayor valor
- `engagement_score`: Mayor uso = mayor retención = mayor LTV

**Acción de Negocio**:
- Priorizar atención a usuarios de alto LTV predicho
- Invertir en retención de segmentos de alto valor
- Optimizar onboarding para aumentar tenure


## Paso 5: Analizar Segmentación


In [ ]:
# Cargar métricas de segmentación
with open("data/models/segmentation_metrics_2026-02-01.json") as f:
    seg_metrics = json.load(f)

print("="*60)
print("MODELO DE SEGMENTACIÓN - MÉTRICAS")
print("="*60)
print(f"Silhouette Score: {seg_metrics['silhouette_score']:.4f}")
print(f"Calinski-Harabasz: {seg_metrics['calinski_harabasz_score']:.2f}")
print(f"Número de Clusters: {seg_metrics['n_clusters']}")

In [ ]:
# Visualizar segmentos
segments_df = pd.DataFrame(seg_metrics['segments'])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Tamaño de segmentos
axes[0, 0].bar(segments_df['segment_name'], segments_df['size'])
axes[0, 0].set_title('Segment Size')
axes[0, 0].set_ylabel('Number of Users')
axes[0, 0].tick_params(axis='x', rotation=45)

# Revenue promedio
axes[0, 1].bar(segments_df['segment_name'], segments_df['avg_revenue'])
axes[0, 1].set_title('Average Revenue by Segment')
axes[0, 1].set_ylabel('Revenue ($)')
axes[0, 1].tick_params(axis='x', rotation=45)

# Engagement promedio
axes[1, 0].bar(segments_df['segment_name'], segments_df['avg_engagement'])
axes[1, 0].set_title('Average Engagement by Segment')
axes[1, 0].set_ylabel('Engagement Score')
axes[1, 0].tick_params(axis='x', rotation=45)

# Tenure promedio
axes[1, 1].bar(segments_df['segment_name'], segments_df['avg_tenure'])
axes[1, 1].set_title('Average Tenure by Segment')
axes[1, 1].set_ylabel('Days')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Tabla de perfiles de segmentos
print("\nPerfiles de Segmentos:\n")
print(segments_df[['segment_name', 'size', 'percentage', 'avg_revenue', 'avg_engagement', 'avg_tenure']].to_string(index=False))

### Interpretación de Segmentos

Típicamente encontramos:

1. **VIP**: Alto revenue, alto engagement, alta tenure
   - Acción: Programa de fidelización premium
   
2. **Engaged**: Buen engagement, tenure media
   - Acción: Upsell a planes superiores
   
3. **At-Risk**: Bajo engagement, payment issues
   - Acción: Campaña de retención, soporte proactivo
   
4. **Inactive**: Muy bajo uso
   - Acción: Win-back campaign o churn prevention


In [ ]:
# Cleanup
spark.stop()

## Paso 6: Generar Reporte de Evaluación

```bash
python -m md_lakehouse.cli evaluate --team configs/teams/team01.yaml --run-date 2026-02-01
```

Esto genera un reporte markdown en `data/reports/evaluation_report_2026-02-01.md` con todos los resultados consolidados.


## Ejercicio Final

**Tarea**:
1. Ejecuta el pipeline completo con `make all`
2. Analiza las métricas de tus modelos
3. Identifica las 3 features más importantes para cada modelo
4. Propón 2 acciones de negocio basadas en cada modelo
5. Compara tus resultados con otro equipo

**Preguntas de reflexión**:
- ¿Por qué las features importantes difieren entre modelos?
- ¿Cómo afecta el churn_base_rate de tu configuración a los resultados?
- ¿Qué pasa si entrenas con diferentes seeds pero mismos parámetros?
